# Intro

## 소리와 음악파일

소리의 3요소: 음량(Loundness), 음고(pitch), 음색(Timbre)

- 음량: 데시벨```[dB]```, 조금 엄밀히 말하면 ```[dB SPL]```(*Sound Pressure Level)
- 음고: 헤르즈```[Hz]```
- 음색: 소리의 색깔, 파형의 모양

## 배음 (Overtone)

피아노에서 친 도와 기타에서 친 도가 왜 다를까? 배음때문이다.

모든 소리는 2차, 3차, 4차 등의 정수배로 배음을 발생한다.

## 컴퓨터로 현악기에서 만들어진 소리를 시뮬레이션 하기 위해서는?

- 위와 같은이유로 컴퓨터에서 친 도와 기타의 도가 다르다
- 기본 주파수 배음을 모두 생성해 주어야 한다. 
- 이것이 카플러스 스트롱 알고리즘

# 카플러스 스트롱 알고리즘

## 이론1: 현악기 시뮬레이션 하기

- 원형버퍼를 만든다 : 기타 줄저럼 양단에 견고하게 묶여있는 줄을 시뮬레이션 한다.
- 원형버퍼의 길이
    - N = S/f (S: 샘플링 레이트, f: 주파수)
- 원형 버퍼의 값들은 [-0.5 ~ 0.5]범위 내의 임의의 값이다.
- 원형버퍼의 값은 감쇠계수(보통0.995)로 줄어든다. (처음 값과 마지막 값의 평균 곱하기 감쇠계수)

In [8]:
#즉 이런 느낌.

N_t1 = [0.1, -0.2, 0.3, 0.6, -0.5]
N_t2 = [-0.2, 0.3, 0.6, -0.5, 0.199]
#N_t2 = [-0.2, 0.3, 0.6, -0.5, 0.1]이 아님. *0.995 * (0.1 +(-0.5)/2)

## 원형 버퍼 구현

In [36]:
from collections import deque
d = deque(range(10))
d

deque([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [37]:
d.append(-1)
d

deque([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, -1])

In [38]:
d.popleft()

0

In [39]:
d

deque([1, 2, 3, 4, 5, 6, 7, 8, 9, -1])

In [41]:
d.rotate()
d

deque([9, -1, 1, 2, 3, 4, 5, 6, 7, 8])

## 이론2 : WAV파일 생성하기

WAV: Waveform Audio Format 파일 오디오 데이터를 저장하는데 사용된다. 
구조가 단순하고 복잡한 압축 기술이 사용되지 않는다. 

기본적으로 비압축 포맷이다. 즉, 소리나는 그대로 값들을 저장해둔 파일이다.

### WAV파일 헤더
- Chunk ID(4byte) - Chunk Size(4byte) - Format(4byte): 
- Chunk ID(4byte) - Chunk Size(4byte) - AudioFormat(2byte)-NumChannels(2byte): 
- Sample Rate (4byte) - Byte Rate(4byte) - Block Align(2byte) - Bits per Samples(2byte): 

==========================================================================================

- Chunk ID : RIFF(Resource Interchange File Format)라고 부르는 ASCII 코드가 있다.
- Sample Rate
    - Number of Samples Per Second. Hz 단위
    - 1초 동안의 소리를 몇 개의 조각으로 쪼개서 저장(분석)하는가..?
    - 10 Hz 라고 하면, 0.1 초 단위로 정보를 저장한다는 뜻. 숫자값이 커질수록 음질이 좋다.
- Byte Rate
    - Average Bytes Per Second. 1초 동안 소리를 내는데 필요한 byte 수! Hz단위
    - 만약 Sample Rate 가 441000 이고, mono 채널이면 기본적으로 1초 재생에 필요한 byte 수는 
        - (Sample 1개가 차지하는 byte) x (1초당 Sample 수) x (채널 수)
        - (Sample 1개가 차지하는 byte) x 441000 x 1
        - 그렇다면 (Sample 1개가 차지하는 byte)는?
        - 'Bits Per Sample' 이다.
- Bit per Sample
    - Sample 한 개를 몇 bit 로 나타낼 것인가라는 의미 다른말로 Bit Depth 라고도 한다.
    - 예를 들어, '도레미파솔라시도' 는 총 음이 8가지다. 이 경우는 2의 3승(8)이고 비트로 나타내면 3 이다.
    - 이 값이 8 이라면, 2의 8승인 256 이 되겠네요. 순간의 소리를 256 레벨로 표현한다.
    - 당연히 이 값이 클수록 음질이 좋다.


### WAV 파일 만들기

In [51]:
import numpy as np
import wave, math

sRate = 44100 #음악 CD가 1초에 읽히는 횟수! 단위는 [Hz]
nSamples = sRate * 5 #5초 짜리 오디오 클립 생성.
x = np.arange(nSamples)/float(sRate) # i /R (샘플 / rate)
vals = np.sin(2.0*math.pi*220.0*x) # A= sin(2pift), A= 진폭, f= 주파수, t=시간, 일단 220Hz의 정현파 만들어 보기

In [52]:
data = np.array(vals*32767, 'int16').tostring() #16비트로 바꾸고 파일에 기록될 수 있도록 문자열로 형변환해준다.
files = wave.open('sine220.wave', 'wb') #wave파일로 저장
files.setparams((1, 2, sRate, nSamples, 'NONE', 'uncompressed')) 
#wave파일 생성에 필요한 매개변수 설정: 단일채널(모노), 2바이트(16비트), 비압축 포맷으로 설정
files.writeframes(data)
files.close()

## 단 5음계

- 음계(musical scale): 음고(주파수)가 증가 혹은 감소하는 순서로 일련의 음을 차례대로 배열한것.
- 음정(musical interval): 두 음고 간의 차이. 일반적으로 작곡을 할 때 모든 음은 특정한 음계에서 선택된다. 
- 반음(semitone): 음계의 기본적인 구성 단위로 서양 음악에서 가장 작은 음정
- 온음(tone): 반음의 2배 길이
- 장음계(major scale): 가장 일반적인 음계의 하나 온음-온음-반음-온음-온음-온음-반음 패턴

5음계 (pentatonic scale) : 5음으로 구성된 음계로 "오! 수잔나" 라는 노래는 5음계를 기반으로 한다.

이 음계의 변형이 단 5음계

(온음+반음)-온음-온음-(온음+반음)-온음

C단조 5음계는: 
 - C 
 - 내림 E 
 - F 
 - G 
 - 내림 B

In [54]:
pentatonic_scale = {"C4":261.6, 
                    "flat E": 311.1,
                    "F": 349.2,
                    "G": 392.0,
                    "flat B": 466.2}

# 카플러스 스트롱 알고리즘 구현

- 1단계: (현악기의 모형이 되는) 원형 버퍼를 구현하고, (deque)으로
- 2단계: 카플러스 스트롱 알고리즘을 이용해서 주어진 주파수의 음들을 생성하고
- 3단계: WAV파일로 기록하고
- 4단계: Python에서 재생하기 위해 pygame으로 WAV파일 재생한다.

## 1단계: deque구현

In [58]:
from collections import deque
import random

buf = deque([random.random() - 0.5 for i in range(10)])
buf

deque([-0.21995300753689306,
       -0.08182415202727578,
       -0.3099845535868049,
       0.0052494446946140805,
       -0.3300677805523089,
       0.4359838853679303,
       0.23057633553293944,
       0.47001291397372347,
       -0.21521278856304804,
       0.029719017620997046])

## 2단계: 카플러스 스트롱 알고리즘 구현

In [59]:
def generateNote(freq):
    nSamples = 44100
    sampleRate = 44100
    N = int(sampleRate/freq)
    buf = deque([random.random() - 0.5 for i in range(N)])
    samples = np.array([0]*nSamples, 'float32') #sample버퍼 초기화
    for i in range(nSamples):
        samples[i] = buf[0]
        avg = 0.996*0.5*(buf[0]+buf[1])
        buf.append(avg)
        buf.popleft()
    samples = np.array(samples*32767, 'int16')
    return samples.tostring()

## 3단계: 오디오 데이터 WAV파일에 기록

In [60]:
def writeWAVE(fname, data):
    files = wave.open(fname, 'wb')
    # WAV 파일 매개변수
    nChannels = 1
    sampleWidth = 2
    frameRate = 44100
    nFrames = 44100
    files.setparams((nChannels,sampleWidth,frameRate,nFrames,'NONE','noncompressed'))
    files.writeframes(data)
    files.close()

## 4단계: Pygame으로 WAV파일 재생하기 

In [61]:
class NotePlayer:
    def __init__(self):
        """44.1kHz의 샘플링 레이트, 16비트 (부호 있음), 단일 채널, 버퍼크기 2048로 pygame의 mixer클래스를 초기화 한다."""
        pygame.mixer.pre_init(44100, -16, 1, 2048)
        pygame.init()
        self.notes = {} # 음으로 이뤄진 딕셔너리
    def add(self, fileName): #음 추가
        """소래 객체를 생성하고, 음 딕셔너리(notes)에 저장한다."""
        self.notes[fileName] = pygame.mixer.Sound(fileName)
    def play(self, fileName): #음 재생
        try:
            self.notes[fileName].play()
        except:
            print(fileName + 'not found!')